In [1]:
%matplotlib inline
%load_ext autoreload
%autoreload 2
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import math
from math import sqrt
import ROOT
import ctypes
try:
#     plt.style.use('belle2')
    plt.style.use('belle2_serif')
#     plt.style.use('belle2_modern')
except OSError:
    print("Please install belle2 matplotlib style") 
px = 1/plt.rcParams['figure.dpi']

from main.data_tools.extract_ntuples import get_pd, get_np
from main.draw_tools.decorations import b2helix, watermark
from main.draw_tools.stacking_with_error_bars import MC_stack_plot, MC_stack_plot_density

from main.data_tools.error_bars import make_data_weight
from main.data_tools.query_dataframes import cut_dfs_7types

from matplotlib.ticker import ScalarFormatter


Welcome to JupyROOT 6.26/04


In [2]:
from math import sqrt

# Error-weighted combination function
def combine_error_weighted(x, y, x_err, y_err):
    central_value = (x / x_err**2 + y / y_err**2) / (1 / x_err**2 + 1 / y_err**2)
    error = 1 / sqrt(1 / x_err**2 + 1 / y_err**2)
    print(f"val 1 = {x}")
    print(f"central value = {central_value*100:.4f}% \pm {error*100:.4f}%")
    return central_value, error

def combine_x_plus_y_divided_by_2(x, y, x_err, y_err):
    central_value = (x+y)/2
    error = sqrt(x_err**2 +  y_err**2)/2
    print(f"val 1 = {x*100:.4f}% \pm {x_err*100:.4f}%")
    print(f"val 2 = {y*100:.4f}% \pm {y_err*100:.4f}%")
    print(f"central value = {central_value*100:.4f}% \pm {error*100:.4f}%")
    return central_value, error

def correct_Acp_stats_no_Kmix( Araw, Araw_err, Aref, Aref_err, Aref_pdg):
    final_Acp = Araw - Aref + Aref_pdg 
    final_Acp_err = sqrt(Araw_err**2 + Aref_err**2)
    print(f"central value = {final_Acp*100:.4f}% \pm {final_Acp_err*100:.4f}%")
    return final_Acp, final_Acp_err

In [3]:
def delta_Acp_sys_unc(A_original, A_original_error, A, A_error):
    delta_Acp = A - A_original
    if A_original_error > A_error:
        delta_Acp_error = sqrt(A_original_error**2 - A_error**2)
    elif A_original_error < A_error:
        delta_Acp_error = sqrt(A_error**2 - A_original_error**2)
    elif A_original_error == A_error:
        delta_Acp_error = 0
    else: 
        print("Error: unable to proceed.")
        sys.exit()

    # print(f"delta_Acp: {delta_Acp}, delta_Acp_error: {delta_Acp_error}")
    print(f"Original Acp: {A_original * 100:.5f}%, Original Acp error: {A_original_error * 100:.5f}%")
    print(f"Acp: {A * 100:.5f}%, Acp error: {A_error * 100:.5f}%")
    print(f"delta_Acp: {delta_Acp * 100:.5f}%, delta_Acp_error: {delta_Acp_error * 100:.5f}%")
    return delta_Acp, delta_Acp_error

In [4]:
#Ks, etapip_gg, fitv3 orig.
central_value_1 =  -0.00495716868227114 
stat_unc_1 =  0.0012428556744885479

central_value_2 =  0.009190873073308792
stat_unc_2 =  0.001354329439157585

combined_central_value, combined_error = combine_x_plus_y_divided_by_2(central_value_1, central_value_2, stat_unc_1, stat_unc_2)

val 1 = -0.4957% \pm 0.1243%
val 2 = 0.9191% \pm 0.1354%
central value = 0.2117% \pm 0.0919%


In [5]:
#Ks, etapip_gg, fitv3 1poly.
central_value_1 =  -0.004940340920340458
stat_unc_1 =  0.001246968375364047

central_value_2 = 0.009189456030007648
stat_unc_2 =  0.0013573688234994894

combined_central_value, combined_error = combine_x_plus_y_divided_by_2(central_value_1, central_value_2, stat_unc_1, stat_unc_2)

val 1 = -0.4940% \pm 0.1247%
val 2 = 0.9189% \pm 0.1357%
central value = 0.2125% \pm 0.0922%


In [6]:
delta_Acp_sys_unc(0.002117, 0.000919, 0.002125, 0.000922)

Original Acp: 0.21170%, Original Acp error: 0.09190%
Acp: 0.21250%, Acp error: 0.09220%
delta_Acp: 0.00080%, delta_Acp_error: 0.00743%


(8.000000000000194e-06, 7.431688906298408e-05)

In [7]:
#Ks, etapip_gg, fitv3 CB.
central_value_1 = -0.004958473252879458
stat_unc_1 =  0.0012427541043529833

central_value_2 = 0.009194674184219265
stat_unc_2 =  0.001352793447416456

combined_central_value, combined_error = combine_x_plus_y_divided_by_2(central_value_1, central_value_2, stat_unc_1, stat_unc_2)

val 1 = -0.4958% \pm 0.1243%
val 2 = 0.9195% \pm 0.1353%
central value = 0.2118% \pm 0.0918%


In [8]:
delta_Acp_sys_unc(0.002117, 0.000919, 0.002118, 0.000918)

Original Acp: 0.21170%, Original Acp error: 0.09190%
Acp: 0.21180%, Acp error: 0.09180%
delta_Acp: 0.00010%, delta_Acp_error: 0.00429%


(1.0000000000001327e-06, 4.286023798347482e-05)

In [9]:
#Ks, etapip_pipipi, fitv3 orig.
central_value_1 =  -0.005620825562001497
stat_unc_1 =   0.0011461459010934164

central_value_2 =  0.009010943177567787
stat_unc_2 =  0.0012540251077938882

combined_central_value, combined_error = combine_x_plus_y_divided_by_2(central_value_1, central_value_2, stat_unc_1, stat_unc_2)

val 1 = -0.5621% \pm 0.1146%
val 2 = 0.9011% \pm 0.1254%
central value = 0.1695% \pm 0.0849%


In [10]:
#Ks, etapip_pipipi, fitv3 1poly.
central_value_1 =  -0.0056210655994511916
stat_unc_1 =  0.0011489942269974941

central_value_2 = 0.0090077710765748
stat_unc_2 =  0.0012556147032909324

combined_central_value, combined_error = combine_x_plus_y_divided_by_2(central_value_1, central_value_2, stat_unc_1, stat_unc_2)

val 1 = -0.5621% \pm 0.1149%
val 2 = 0.9008% \pm 0.1256%
central value = 0.1693% \pm 0.0851%


In [11]:
delta_Acp_sys_unc(0.001695, 0.000849, 0.001693, 0.000851)

Original Acp: 0.16950%, Original Acp error: 0.08490%
Acp: 0.16930%, Acp error: 0.08510%
delta_Acp: -0.00020%, delta_Acp_error: 0.00583%


(-1.9999999999998318e-06, 5.830951894845156e-05)

In [12]:
#Ks, etapip_pipipi, fitv3 CB.
central_value_1 = -0.005614514717638852
stat_unc_1 =  0.0011424497820504143

central_value_2 = 0.009003726336860973
stat_unc_2 =  0.001248844450761455

combined_central_value, combined_error = combine_x_plus_y_divided_by_2(central_value_1, central_value_2, stat_unc_1, stat_unc_2)

val 1 = -0.5615% \pm 0.1142%
val 2 = 0.9004% \pm 0.1249%
central value = 0.1695% \pm 0.0846%


In [13]:
delta_Acp_sys_unc(0.001695, 0.000849, 0.001695, 0.000846)

Original Acp: 0.16950%, Original Acp error: 0.08490%
Acp: 0.16950%, Acp error: 0.08460%
delta_Acp: 0.00000%, delta_Acp_error: 0.00713%


(0.0, 7.130918594402955e-05)

In [14]:
#Ks_K, etapip_gg_K, fitv3 orig.
central_value_1 =  -0.000891053282276677
stat_unc_1 = 0.002591316856349276

central_value_2 = 0.01696678351479708  
stat_unc_2 = 0.002695345787618729

combined_central_value, combined_error = combine_x_plus_y_divided_by_2(central_value_1, central_value_2, stat_unc_1, stat_unc_2)

val 1 = -0.0891% \pm 0.2591%
val 2 = 1.6967% \pm 0.2695%
central value = 0.8038% \pm 0.1869%


In [15]:
#Ks_K, etapip_gg_K, fitv3 1poly.
central_value_1 =  -0.0008901233494843508
stat_unc_1 = 0.0025924380483908463

central_value_2 =  0.016973297215573835
stat_unc_2 = 0.0026966559203443967

combined_central_value, combined_error = combine_x_plus_y_divided_by_2(central_value_1, central_value_2, stat_unc_1, stat_unc_2)

val 1 = -0.0890% \pm 0.2592%
val 2 = 1.6973% \pm 0.2697%
central value = 0.8042% \pm 0.1870%


In [16]:
delta_Acp_sys_unc(0.008038 , 0.001869, 0.008042, 0.001870)

Original Acp: 0.80380%, Original Acp error: 0.18690%
Acp: 0.80420%, Acp error: 0.18700%
delta_Acp: 0.00040%, delta_Acp_error: 0.00611%


(4.000000000000531e-06, 6.114736298483902e-05)

In [17]:
#Ks_K, etapip_gg_K, fitv3 CB.
central_value_1 =  -0.0008873317613541376
stat_unc_1 = 0.002602211387280728

central_value_2 =  0.016955134844759856
stat_unc_2 = 0.002712599952763657

combined_central_value, combined_error = combine_x_plus_y_divided_by_2(central_value_1, central_value_2, stat_unc_1, stat_unc_2)

val 1 = -0.0887% \pm 0.2602%
val 2 = 1.6955% \pm 0.2713%
central value = 0.8034% \pm 0.1879%


In [18]:
delta_Acp_sys_unc(0.008038 , 0.001869,  0.008034, 0.001879)

Original Acp: 0.80380%, Original Acp error: 0.18690%
Acp: 0.80340%, Acp error: 0.18790%
delta_Acp: -0.00040%, delta_Acp_error: 0.01936%


(-4.000000000000531e-06, 0.00019359752064528193)

In [19]:
#Ks_K, etapip_pipipi_K, fitv3 orig.
central_value_1 =  0.00013218589725561003
stat_unc_1 = 0.002166365637515913

central_value_2 =  0.01753684985818671
stat_unc_2 = 0.0022719620271573465

combined_central_value, combined_error = combine_x_plus_y_divided_by_2(central_value_1, central_value_2, stat_unc_1, stat_unc_2)

val 1 = 0.0132% \pm 0.2166%
val 2 = 1.7537% \pm 0.2272%
central value = 0.8835% \pm 0.1570%


In [20]:
#Ks_K, etapip_pipipi_K, fitv3 1poly.
central_value_1 =  0.00015186157220825613 
stat_unc_1 = 0.0021684634044209523

central_value_2 =  0.0175525946890438
stat_unc_2 = 0.0022736901600101922

combined_central_value, combined_error = combine_x_plus_y_divided_by_2(central_value_1, central_value_2, stat_unc_1, stat_unc_2)

val 1 = 0.0152% \pm 0.2168%
val 2 = 1.7553% \pm 0.2274%
central value = 0.8852% \pm 0.1571%


In [21]:
delta_Acp_sys_unc(0.008835 , 0.001570, 0.008852, 0.001571)

Original Acp: 0.88350%, Original Acp error: 0.15700%
Acp: 0.88520%, Acp error: 0.15710%
delta_Acp: 0.00170%, delta_Acp_error: 0.00560%


(1.6999999999999654e-05, 5.604462507680527e-05)

In [22]:
#Ks_K, etapip_pipipi_K, fitv3 CB.
central_value_1 =  0.00013496170614857306
stat_unc_1 = 0.002166978507845386

central_value_2 =  0.017531594361358538
stat_unc_2 = 0.0022843644090084006

combined_central_value, combined_error = combine_x_plus_y_divided_by_2(central_value_1, central_value_2, stat_unc_1, stat_unc_2)

val 1 = 0.0135% \pm 0.2167%
val 2 = 1.7532% \pm 0.2284%
central value = 0.8833% \pm 0.1574%


In [23]:
delta_Acp_sys_unc(0.008835 , 0.001570, 0.008833 , 0.001574)

Original Acp: 0.88350%, Original Acp error: 0.15700%
Acp: 0.88330%, Acp error: 0.15740%
delta_Acp: -0.00020%, delta_Acp_error: 0.01121%


(-2.0000000000002655e-06, 0.00011214276615101076)